# 00-1. Preprocessing (CPTAC-COAD)

In [17]:
library(data.table)
library(vespa)

In [18]:
if (!dir.exists("./data/cptac-coad")) {
  out <- system2(
    "bash",
    "./tools/scripts/cptac-coad.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  4.42M 100  4.42M   0      0  3.61M      0   00:01   00:01          1.49M
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  3.36M 100  3.36M   0      0  2.93M      0   00:01   00:01          1.36M
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  19263 100  19263   0      0  37954      0                              0


In [19]:
if (!file.exists("./tools/references/library.fasta")) {
  out <- system2(
    "bash",
    "./tools/scripts/fasta.sh",
    stdout = TRUE,
    stderr = TRUE
  )

  cat(out, sep = "\n")
}

## Import data

In [21]:
proteo <- fread(
    "./data/cptac-coad/Human__CPTAC_COAD__PNNL__Proteome__TMT__03_01_2017__BCM__Gene__PNNL_Tumor_TMT_UnsharedLogRatio.cct"
)
proteo

attrib_name,01CO005,01CO006,01CO008,01CO013,01CO014,01CO015,01CO019,01CO022,05CO002,⋯,20CO001,20CO003,20CO004,20CO006,20CO007,21CO006,21CO007,22CO004,22CO006,27CO004
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,-1.1000,-1.1200,-1.2000,-1.8900,-0.5230,-1.6200,-0.3110,0.9060,-1.1400,⋯,0.6290,-0.1230,-1.8700,-0.3680,-1.0400,-0.5570,-0.9750,-1.2800,0.5150,-1.6400
A1CF,0.3180,-0.4410,0.1600,0.1120,-0.2480,0.2630,-0.2700,-0.7780,0.3850,⋯,-0.0680,-1.0700,0.4610,0.9170,-0.3830,-0.4470,0.3980,0.5180,-1.2100,-0.0693
A2M,-0.4870,-0.3470,-1.8500,-0.3290,-0.6380,-0.9760,-0.9210,1.4500,-1.3700,⋯,0.0734,-0.4520,-1.8500,-0.4040,-1.6600,0.3420,-1.1500,-0.8720,-0.0283,-0.2540
AAAS,0.0995,-0.0029,0.1190,0.6700,0.2890,0.5220,0.2260,0.3590,0.0167,⋯,-0.6540,0.2660,-0.1570,0.1420,0.0356,-0.1390,0.5730,0.2580,0.1780,0.0520
AACS,0.1550,0.0957,-0.0924,0.1160,0.3780,-0.2730,0.0528,0.2190,0.2120,⋯,0.2130,-0.2280,1.2900,0.0688,0.1530,0.1400,-0.1280,0.4240,-0.3030,-0.1370
AAGAB,0.1690,0.3960,0.0187,0.3130,0.8220,0.5040,0.0428,0.7710,-0.0213,⋯,0.1550,0.0484,0.1330,-0.0619,0.0100,0.0517,0.0600,0.2980,0.1500,0.1450
AAK1,0.0653,-0.0363,-0.2140,-0.2380,0.0018,-0.3180,-0.0620,-0.0471,-0.1430,⋯,0.5650,0.1710,-0.2230,0.2170,-0.2210,0.0039,-0.0559,-0.1500,0.3990,0.1100
AAMDC,-0.1470,-0.5490,0.3280,-0.2740,-1.0800,-0.8460,-0.1240,-0.9160,-0.9580,⋯,-0.0406,0.7850,-0.2460,-0.1520,-0.8930,-0.1180,-1.3300,-0.6580,-1.0000,0.1400
AAMP,0.1140,0.2200,-0.2820,-0.5540,0.4980,0.2000,0.0779,0.4770,0.0682,⋯,-0.1540,0.3000,0.2350,-0.0663,-0.2530,0.1020,-0.3650,0.0066,0.2060,-0.5690


In [22]:
setnames(proteo, "attrib_name", "gene_id")

fasta_headers <- readLines("./tools/references/library.fasta")
fasta_headers <- fasta_headers[grepl("^>", fasta_headers)]

protein_id <- sub("^>[^|]*\\|([^|]+)\\|.*", "\\1", fasta_headers)

gene_symbol <- ifelse(
  grepl(" GN=", fasta_headers),
  sub(".* GN=([^ ]+).*", "\\1", fasta_headers),
  NA_character_
)

map_dt <- data.table(
  gene_id = gene_symbol,
  protein_id = protein_id,
  fasta_header = fasta_headers
)

map_dt <- map_dt[!is.na(gene_id)]

map_dt[, is_sp := grepl("^>sp\\|", fasta_header)]
setorder(map_dt, gene_id, -is_sp)

map_dt <- map_dt[, .SD[1], by = gene_id]
map_dt <- map_dt[, .(gene_id, protein_id)]

proteo_annot <- merge(
  proteo,
  map_dt,
  by = "gene_id",
  all.x = TRUE
)

cat("genes in proteo:", nrow(proteo_annot), "\n")
cat("mapped genes:", sum(!is.na(proteo_annot$protein_id)), "\n")
cat("unmapped genes:", sum(is.na(proteo_annot$protein_id)), "\n")

genes in proteo: 8067 
mapped genes: 7859 
unmapped genes: 208 


In [24]:
proteo_annot <- proteo_annot[!is.na(protein_id)]

sample_cols <- setdiff(colnames(proteo_annot), c("gene_id", "protein_id"))

proteo_long <- melt(
  proteo_annot,
  id.vars = c("gene_id", "protein_id"),
  measure.vars = sample_cols,
  variable.name = "run_id",
  value.name = "peptide_intensity"
)

proteo_long[, peptide_id := protein_id]
proteo_long[, modified_peptide_sequence := protein_id]
proteo_long[, peptide_sequence := protein_id]
proteo_long[, phosphosite := "PA"]
proteo_long[, site_id := paste(gene_id, protein_id, phosphosite, sep = ":")]

proteo_long <- proteo_long[
  ,
  .(
    gene_id,
    protein_id,
    peptide_id,
    site_id,
    modified_peptide_sequence,
    peptide_sequence,
    phosphosite,
    run_id,
    peptide_intensity
  )
]

dim(proteo_long)
head(proteo_long)

[1] 762323      9

gene_id,protein_id,peptide_id,site_id,modified_peptide_sequence,peptide_sequence,phosphosite,run_id,peptide_intensity
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<dbl>
A1BG,P04217,P04217,A1BG:P04217:PA,P04217,P04217,PA,01CO005,-1.1000
A1CF,Q9NQ94,Q9NQ94,A1CF:Q9NQ94:PA,Q9NQ94,Q9NQ94,PA,01CO005,0.3180
A2M,P01023,P01023,A2M:P01023:PA,P01023,P01023,PA,01CO005,-0.4870
AAAS,Q9NRG9,Q9NRG9,AAAS:Q9NRG9:PA,Q9NRG9,Q9NRG9,PA,01CO005,0.0995
AACS,Q86V21,Q86V21,AACS:Q86V21:PA,Q86V21,Q86V21,PA,01CO005,0.1550
AAGAB,Q6PD74,Q6PD74,AAGAB:Q6PD74:PA,Q6PD74,Q6PD74,PA,01CO005,0.1690


In [25]:
phospho <- fread(
    "./data/cptac-coad/Human__CPTAC_COAD__PNNL__Phosphoproteome__TMT__03_01_2017__BCM__Site__Tumor_PNNL_TMT_LogRatio.cct.gz"
)
phospho

Warning message in fread("./data/cptac-coad/Human__CPTAC_COAD__PNNL__Phosphoproteome__TMT__03_01_2017__BCM__Site__Tumor_PNNL_TMT_LogRatio.cct.gz"):
“Detected 97 column names but the data has 98 columns (i.e. invalid file). Added an extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”


V1,01CO005,01CO006,01CO008,01CO013,01CO014,01CO015,01CO019,01CO022,05CO002,⋯,20CO001,20CO003,20CO004,20CO006,20CO007,21CO006,21CO007,22CO004,22CO006,27CO004
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
SLC12A8_T485__A0AV02:T485,NA,1.1050,NA,NA,0.606,NA,NA,0.5350,NA,⋯,NA,-0.8280,0.5610,NA,NA,NA,NA,-1.227,NA,NA
E2F8_S102__A0AVK6:S102,NA,NA,NA,NA,NA,NA,NA,0.4750,NA,⋯,0.5610,NA,NA,NA,NA,NA,NA,-0.453,NA,NA
E2F8_S413__A0AVK6:S413,NA,NA,NA,-0.3720,NA,NA,NA,NA,NA,⋯,0.4760,NA,NA,NA,NA,NA,NA,NA,NA,-0.2210
E2F8_S417__A0AVK6:S417,NA,NA,NA,0.0590,NA,NA,NA,NA,NA,⋯,0.4760,NA,NA,NA,NA,NA,NA,NA,NA,-0.2265
E2F8_S664__A0AVK6:S664,NA,NA,NA,NA,0.160,NA,NA,0.4190,NA,⋯,-1.1020,NA,0.3540,NA,NA,NA,NA,NA,NA,NA
E2F8_S71__A0AVK6:S71,NA,-0.2520,-0.9420,0.5350,NA,0.7020,-0.1300,NA,0.3080,⋯,0.6920,NA,NA,-0.2180,NA,NA,NA,NA,-0.2680,-0.0700
E2F8_T58__A0AVK6:T58,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
UBA6_S36__A0AVT1:S36,-0.6170,NA,NA,NA,NA,0.1140,NA,-0.2070,NA,⋯,NA,-0.2280,NA,NA,-0.7180,NA,NA,-0.248,NA,NA
ESYT2_S513__A0FGR8:S513,NA,NA,NA,-1.1360,0.033,-0.4990,NA,NA,NA,⋯,-0.2550,0.7370,0.1310,NA,NA,NA,NA,NA,NA,-0.4330
